# 🛠️ FraudLens — Feature Engineering Pipeline (02)

**Goal:** Apply the functions from `src/features.py` to build production-ready
feature tables for both datasets, verify no data leakage, and save outputs to
`data/processed/`.

> ⚠️ IEEE-CIS and PaySim are kept **completely separate** throughout.
  They have different schemas and must never be merged.

---

## 0. Setup & Imports

In [5]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
matplotlib.rcParams.update({'figure.dpi': 110, 'font.family': 'DejaVu Sans'})
sns.set_theme(style='whitegrid', palette='muted')

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR      = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR      = os.path.join(BASE_DIR, 'data')
IEEE_DIR      = os.path.join(DATA_DIR, 'ieee-cis')
PAYSIM_DIR    = os.path.join(DATA_DIR, 'paysim')
PROCESSED_DIR = os.path.join(DATA_DIR, 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

# ── Import feature engineering library ─────────────────────────────────────
sys.path.insert(0, BASE_DIR)
from src.features import (
    decode_transaction_dt,
    log_transform_amount,
    FrequencyEncoder,
    add_has_identity,
    add_missingness_flags,
    drop_high_missing_v_features,
    prune_correlated_v_features,
    add_entity_graph_features,
    add_paysim_features,
    time_split_ieee,
    time_split_paysim,
    get_v_cols,
    get_id_cols,
)

print(f'Base dir      : {BASE_DIR}')
print(f'Processed dir : {PROCESSED_DIR}')
print(f'Pandas        : {pd.__version__}  |  NumPy : {np.__version__}')


Base dir      : c:\Users\DHANRAJ\fraudlens
Processed dir : c:\Users\DHANRAJ\fraudlens\data\processed
Pandas        : 3.0.3  |  NumPy : 2.4.4


---
# Part 1 — IEEE-CIS Feature Engineering

Pipeline steps:
1. Load raw CSVs
2. Decode TransactionDT
3. Log-transform TransactionAmt
4. Add has_identity flag
5. Add missingness flags (V-features + identity cols with >30% missing)
6. Drop V-features with >90% missing
7. Frequency-encode card1, addr1, P_emaildomain (fit on train only)
8. Prune correlated V-features (|r| > 0.95, keep fewer-NaN)
9. Add entity-graph aggregation features
10. Time-based split (70/15/15)
11. Save to parquet


## 1.1 Load IEEE-CIS Raw Data

In [6]:
print('Loading train_transaction.csv (may take ~30s) ...')
txn = pd.read_csv(os.path.join(IEEE_DIR, 'train_transaction.csv'))
print(f'  train_transaction shape : {txn.shape}')

print('Loading train_identity.csv ...')
idn = pd.read_csv(os.path.join(IEEE_DIR, 'train_identity.csv'))
print(f'  train_identity    shape : {idn.shape}')

# Left-join: transaction <- identity
ieee = txn.merge(idn, on='TransactionID', how='left')
print(f'\nMerged IEEE-CIS shape : {ieee.shape}')
print(f'  Fraud rate          : {ieee["isFraud"].mean():.4%}')


Loading train_transaction.csv (may take ~30s) ...
  train_transaction shape : (590540, 394)
Loading train_identity.csv ...
  train_identity    shape : (144233, 41)

Merged IEEE-CIS shape : (590540, 434)
  Fraud rate          : 3.4990%


## 1.2 Pre-Split Transformations

Steps that do **not** require fitting (no leakage risk):
- Decode TransactionDT
- Log-transform TransactionAmt
- Add has_identity flag


In [7]:
shape_before = ieee.shape

# 1. Decode TransactionDT
ieee = decode_transaction_dt(ieee)
print(f'After decode_transaction_dt : {ieee.shape}  (added hour_of_day, day_of_week, day_of_month)')

# 2. Log-transform TransactionAmt
ieee = log_transform_amount(ieee)
print(f'After log_transform_amount  : {ieee.shape}  (added TransactionAmt_log1p)')

# 3. has_identity flag — identity_ids comes from the identity table (no leakage: it's
#    always the full train_identity.csv, which we have.  At inference time, the
#    scoring service checks the identity lookup in real time.)
ieee = add_has_identity(ieee, idn['TransactionID'])
print(f'After add_has_identity       : {ieee.shape}  (added has_identity)')
print(f'  Transactions with identity : {ieee["has_identity"].sum():,}  ({ieee["has_identity"].mean():.1%})')


After decode_transaction_dt : (590540, 437)  (added hour_of_day, day_of_week, day_of_month)
After log_transform_amount  : (590540, 438)  (added TransactionAmt_log1p)
After add_has_identity       : (590540, 439)  (added has_identity)
  Transactions with identity : 144,233  (24.4%)


## 1.3 Chronological Split (70 / 15 / 15)

**Important:** We split by TransactionDT *before* any fitting steps
(frequency encoding, correlation pruning) so those steps can be fitted
exclusively on the training portion.


In [8]:
ieee_train, ieee_val, ieee_test, ieee_split_info = time_split_ieee(ieee)

print(f'\nIEEE-CIS split summary:')
print(f'  Train : {len(ieee_train):>8,} rows   fraud: {ieee_train["isFraud"].mean():.4%}')
print(f'  Val   : {len(ieee_val):>8,} rows   fraud: {ieee_val["isFraud"].mean():.4%}')
print(f'  Test  : {len(ieee_test):>8,} rows   fraud: {ieee_test["isFraud"].mean():.4%}')



  [ieee split]  Sorting by TransactionDT and splitting 70/15/15 ...
    Rows   -> train:  413,378  val:   88,581  test:   88,581  (total: 590,540)
      TransactionDT range -> train [86400 - 10437996]  val [10438003 - 13151840]  test [13151880 - 15811131]
    Confirmed: no temporal overlap between splits.

IEEE-CIS split summary:
  Train :  413,378 rows   fraud: 3.5169%
  Val   :   88,581 rows   fraud: 3.4341%
  Test  :   88,581 rows   fraud: 3.4804%


## 1.4 Missingness Flags (fit on train, apply everywhere)

In [9]:
v_cols_all  = get_v_cols(ieee_train)
id_cols_all = get_id_cols(ieee_train)
print(f'V-feature columns  : {len(v_cols_all)}')
print(f'Identity columns   : {len(id_cols_all)}')

# Add flags — reference is always the training split
ieee_train = add_missingness_flags(ieee_train, v_cols_all, id_cols_all,
                                    reference_df=ieee_train)
ieee_val   = add_missingness_flags(ieee_val,   v_cols_all, id_cols_all,
                                    reference_df=ieee_train)
ieee_test  = add_missingness_flags(ieee_test,  v_cols_all, id_cols_all,
                                    reference_df=ieee_train)
print(f'Train shape after flags: {ieee_train.shape}')


V-feature columns  : 339
Identity columns   : 38
  [missingness flags] Added 208 flag columns (threshold >30%).
  [missingness flags] Added 208 flag columns (threshold >30%).
  [missingness flags] Added 208 flag columns (threshold >30%).
Train shape after flags: (413378, 647)


## 1.5 Drop V-Features with >90% Missing

In [10]:
ieee_train, dropped_90 = drop_high_missing_v_features(
    ieee_train, v_cols_all, reference_df=ieee_train
)
# Apply same drops to val and test
ieee_val  = ieee_val.drop(columns=[c for c in dropped_90 if c in ieee_val.columns])
ieee_test = ieee_test.drop(columns=[c for c in dropped_90 if c in ieee_test.columns])

# Refresh V-col list after dropping
v_cols_after_drop = get_v_cols(ieee_train)
print(f'\nV-features remaining after >90%% drop: {len(v_cols_after_drop)}')
print(f'Train shape: {ieee_train.shape}')



  [V-feature drop >90% missing]  Dropped 0 V-features:
    (none)

V-features remaining after >90%% drop: 339
Train shape: (413378, 647)


## 1.6 Frequency Encoding (fit on train only)

In [11]:
enc = FrequencyEncoder()   # encodes card1, addr1, P_emaildomain
ieee_train = enc.fit_transform(ieee_train)
ieee_val   = enc.transform(ieee_val)
ieee_test  = enc.transform(ieee_test)

print('Frequency encoder fitted on training split.')
for col, fmap in enc.freq_maps_.items():
    print(f'  {col:>15s}  unique values: {len(fmap):,}')
print(f'Train shape: {ieee_train.shape}')


Frequency encoder fitted on training split.
            card1  unique values: 12,242
            addr1  unique values: 318
    P_emaildomain  unique values: 59
Train shape: (413378, 650)


## 1.7 V-Feature Correlation Pruning (fit on train)

For each pair of V-features with Pearson |r| > 0.95, the one with more
missing values is dropped.  Ties broken lexicographically.


In [12]:
ieee_train, dropped_corr = prune_correlated_v_features(
    ieee_train, v_cols_after_drop, reference_df=ieee_train
)
# Apply same drops to val and test
ieee_val  = ieee_val.drop(columns=[c for c in dropped_corr if c in ieee_val.columns])
ieee_test = ieee_test.drop(columns=[c for c in dropped_corr if c in ieee_test.columns])

v_cols_final = get_v_cols(ieee_train)
print(f'\nFinal V-feature count: {len(v_cols_final)}')
print(f'Train shape: {ieee_train.shape}')



  [V-feature correlation pruning]  Starting with 339 V-features.
    Computing correlation matrix (may take a moment on large data)...
    Dropped 76 correlated V-features (|r| > 0.95).
    Remaining V-features: 263  (was 339).

Final V-feature count: 263
Train shape: (413378, 574)


## 1.8 Entity-Graph Aggregation Features

Aggregations are computed on the **training split only** and then
mapped by key to val/test (no per-row lookahead into future transactions).


In [13]:
ieee_train = add_entity_graph_features(ieee_train, train_df=ieee_train)
ieee_val   = add_entity_graph_features(ieee_val,   train_df=ieee_train)
ieee_test  = add_entity_graph_features(ieee_test,  train_df=ieee_train)

print(f'Train shape: {ieee_train.shape}')
print(f'Val   shape: {ieee_val.shape}')
print(f'Test  shape: {ieee_test.shape}')


  [entity-graph features] Added: card1_addr1_count, card1_addr1_email_count, card1_addr1_amt_mean, card1_addr1_amt_std.
  [entity-graph features] Added: card1_addr1_count, card1_addr1_email_count, card1_addr1_amt_mean, card1_addr1_amt_std.
  [entity-graph features] Added: card1_addr1_count, card1_addr1_email_count, card1_addr1_amt_mean, card1_addr1_amt_std.
Train shape: (413378, 578)
Val   shape: (88581, 578)
Test  shape: (88581, 578)


## 1.9 ✅ No-Leakage Verification

Print the TransactionDT range for each split and confirm no overlap.


In [14]:
print('=== IEEE-CIS TransactionDT ranges (raw seconds elapsed) ===')
for name, df in [('Train', ieee_train), ('Val', ieee_val), ('Test', ieee_test)]:
    mn = df['TransactionDT'].min()
    mx = df['TransactionDT'].max()
    n  = len(df)
    print(f'  {name:5s}: [{mn:>12.0f}  –  {mx:>12.0f}]  ({n:,} rows)')

print()
assert ieee_train['TransactionDT'].max() <= ieee_val['TransactionDT'].min(), \
    'LEAKAGE: train overlaps val!'
assert ieee_val['TransactionDT'].max()   <= ieee_test['TransactionDT'].min(), \
    'LEAKAGE: val overlaps test!'
print('All assertions passed: no temporal overlap between splits.')

print()
print('=== Decoded time-component ranges ===')
for name, df in [('Train', ieee_train), ('Val', ieee_val), ('Test', ieee_test)]:
    print(f'  {name:5s}  hour_of_day: [{df["hour_of_day"].min()}–{df["hour_of_day"].max()}]'
          f'  day_of_week: [{df["day_of_week"].min()}–{df["day_of_week"].max()}]'
          f'  day_of_month: [{df["day_of_month"].min()}–{df["day_of_month"].max()}]')


=== IEEE-CIS TransactionDT ranges (raw seconds elapsed) ===
  Train: [       86400  –      10437996]  (413,378 rows)
  Val  : [    10438003  –      13151840]  (88,581 rows)
  Test : [    13151880  –      15811131]  (88,581 rows)

All assertions passed: no temporal overlap between splits.

=== Decoded time-component ranges ===
  Train  hour_of_day: [0–23]  day_of_week: [0–6]  day_of_month: [1–31]
  Val    hour_of_day: [0–23]  day_of_week: [0–6]  day_of_month: [1–31]
  Test   hour_of_day: [0–23]  day_of_week: [0–6]  day_of_month: [1–31]


## 1.10 Shape Before vs. After

In [15]:
print('=== IEEE-CIS: Shape summary ===')
print(f'  Raw merged            : 590540 rows × 434 cols')
print(f'  After pipeline (train): {ieee_train.shape[0]:,} rows × {ieee_train.shape[1]} cols')
print(f'  After pipeline (val)  : {ieee_val.shape[0]:,} rows × {ieee_val.shape[1]} cols')
print(f'  After pipeline (test) : {ieee_test.shape[0]:,} rows × {ieee_test.shape[1]} cols')

feat_cols = [c for c in ieee_train.columns
             if c not in ('TransactionID', 'isFraud')]
print(f'\n  Feature columns used for modeling: {len(feat_cols)}')


=== IEEE-CIS: Shape summary ===
  Raw merged            : 590540 rows × 434 cols
  After pipeline (train): 413,378 rows × 578 cols
  After pipeline (val)  : 88,581 rows × 578 cols
  After pipeline (test) : 88,581 rows × 578 cols

  Feature columns used for modeling: 576


## 1.11 Save IEEE-CIS Processed Files

In [16]:
ieee_train.to_parquet(os.path.join(PROCESSED_DIR, 'ieee_train.parquet'), index=False)
ieee_val.to_parquet(  os.path.join(PROCESSED_DIR, 'ieee_val.parquet'),   index=False)
ieee_test.to_parquet( os.path.join(PROCESSED_DIR, 'ieee_test.parquet'),  index=False)

for fname in ('ieee_train.parquet', 'ieee_val.parquet', 'ieee_test.parquet'):
    size_mb = os.path.getsize(os.path.join(PROCESSED_DIR, fname)) / 1024**2
    print(f'  Saved {fname:<30s}  ({size_mb:.1f} MB)')


  Saved ieee_train.parquet              (63.9 MB)
  Saved ieee_val.parquet                (14.0 MB)
  Saved ieee_test.parquet               (14.8 MB)


---
# Part 2 — PaySim Feature Engineering

PaySim has **zero missing values** (synthetic data) and a much simpler
schema (11 columns).  All feature engineering is a single stateless pass.


## 2.1 Load PaySim Raw Data

In [17]:
paysim_files = [f for f in os.listdir(PAYSIM_DIR) if f.endswith('.csv')]
print(f'PaySim CSV files found: {paysim_files}')

print('Loading PaySim CSV (may take ~20s) ...')
paysim = pd.read_csv(os.path.join(PAYSIM_DIR, paysim_files[0]))
print(f'Raw PaySim shape : {paysim.shape}')
print(f'Fraud rate       : {paysim["isFraud"].mean():.4%}')
print(f'\nColumns: {paysim.columns.tolist()}')


PaySim CSV files found: ['PS_20174392719_1491204439457_log.csv']
Loading PaySim CSV (may take ~20s) ...
Raw PaySim shape : (6362620, 11)
Fraud rate       : 0.1291%

Columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']


## 2.2 PaySim Feature Engineering

In [18]:
shape_before_ps = paysim.shape

paysim_fe = add_paysim_features(paysim)

print(f'\nShape before : {shape_before_ps}')
print(f'Shape after  : {paysim_fe.shape}')
new_feats = [c for c in paysim_fe.columns if c not in paysim.columns]
print(f'New columns  : {new_feats}')


  [paysim features] Added: balance_delta_orig, is_transfer, is_cashout, orig_balance_drained, dest_balance_zero, amount_to_balance_ratio, step_sin, step_cos.

Shape before : (6362620, 11)
Shape after  : (6362620, 19)
New columns  : ['balance_delta_orig', 'is_transfer', 'is_cashout', 'orig_balance_drained', 'dest_balance_zero', 'amount_to_balance_ratio', 'step_sin', 'step_cos']


## 2.3 Sanity-check New PaySim Features

In [19]:
check_cols = [
    'balance_delta_orig', 'is_transfer', 'is_cashout',
    'orig_balance_drained', 'dest_balance_zero',
    'amount_to_balance_ratio', 'step_sin', 'step_cos',
]
display(paysim_fe[check_cols].describe().round(4))

print('\nFraud rate by transaction type (sanity check):')
display(
    paysim_fe.groupby('type')['isFraud']
    .agg(['sum','count'])
    .assign(fraud_rate=lambda x: x['sum'] / x['count'])
    .sort_values('fraud_rate', ascending=False)
)


,balance_delta_orig,is_transfer,is_cashout,orig_balance_drained,dest_balance_zero,amount_to_balance_ratio,step_sin,step_cos
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,-2.010925e+05,8.380000e-02,3.517000e-01,5.673000e-01,3.834000e-01,7.067448e+04,-4.492000e-01,-3.032000e-01
std,6.066505e+05,2.770000e-01,4.775000e-01,4.954000e-01,4.862000e-01,5.084243e+05,5.475000e-01,6.376000e-01
min,-9.244552e+07,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-1.000000e+00,-1.000000e+00
25%,-2.496411e+05,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.344000e-01,-9.659000e-01,-8.660000e-01
50%,-6.867726e+04,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,6.453800e+00,-7.071000e-01,-5.000000e-01
75%,-2.954230e+03,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.228776e+04,0.000000e+00,2.588000e-01
max,1.000000e-02,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,9.244552e+07,1.000000e+00,1.000000e+00



Fraud rate by transaction type (sanity check):


,sum,count,fraud_rate
type,,,
TRANSFER,4097,532909,0.007688
CASH_OUT,4116,2237500,0.001840
CASH_IN,0,1399284,0.000000
DEBIT,0,41432,0.000000
PAYMENT,0,2151495,0.000000


## 2.4 Chronological Split (70 / 15 / 15)

In [20]:
ps_train, ps_val, ps_test, ps_split_info = time_split_paysim(paysim_fe)

print(f'\nPaySim split summary:')
print(f'  Train : {len(ps_train):>9,} rows   fraud: {ps_train["isFraud"].mean():.4%}')
print(f'  Val   : {len(ps_val):>9,} rows   fraud: {ps_val["isFraud"].mean():.4%}')
print(f'  Test  : {len(ps_test):>9,} rows   fraud: {ps_test["isFraud"].mean():.4%}')



  [paysim split]  Sorting by step and splitting 70/15/15 ...
    Rows   -> train: 4,453,834  val:  954,393  test:  954,393  (total: 6,362,620)
               step range -> train [1 - 323]  val [323 - 378]  test [378 - 743]
    Confirmed: no temporal overlap between splits.

PaySim split summary:
  Train : 4,453,834 rows   fraud: 0.0818%
  Val   :   954,393 rows   fraud: 0.0589%
  Test  :   954,393 rows   fraud: 0.4200%


## 2.5 ✅ No-Leakage Verification

Print the `step` range for each split and confirm no overlap.


In [21]:
print('=== PaySim step ranges ===')
for name, df in [('Train', ps_train), ('Val', ps_val), ('Test', ps_test)]:
    mn = df['step'].min()
    mx = df['step'].max()
    n  = len(df)
    print(f'  {name:5s}: step [{mn:>4d}  –  {mx:>4d}]  ({n:,} rows)')

print()
assert ps_train['step'].max() <= ps_val['step'].min(),  'LEAKAGE: train overlaps val!'
assert ps_val['step'].max()   <= ps_test['step'].min(), 'LEAKAGE: val overlaps test!'
print('All assertions passed: no temporal overlap between splits.')


=== PaySim step ranges ===
  Train: step [   1  –   323]  (4,453,834 rows)
  Val  : step [ 323  –   378]  (954,393 rows)
  Test : step [ 378  –   743]  (954,393 rows)

All assertions passed: no temporal overlap between splits.


## 2.6 Save PaySim Processed Files

In [22]:
ps_train.to_parquet(os.path.join(PROCESSED_DIR, 'paysim_train.parquet'), index=False)
ps_val.to_parquet(  os.path.join(PROCESSED_DIR, 'paysim_val.parquet'),   index=False)
ps_test.to_parquet( os.path.join(PROCESSED_DIR, 'paysim_test.parquet'),  index=False)

for fname in ('paysim_train.parquet', 'paysim_val.parquet', 'paysim_test.parquet'):
    size_mb = os.path.getsize(os.path.join(PROCESSED_DIR, fname)) / 1024**2
    print(f'  Saved {fname:<30s}  ({size_mb:.1f} MB)')


  Saved paysim_train.parquet            (241.8 MB)
  Saved paysim_val.parquet              (52.0 MB)
  Saved paysim_test.parquet             (52.0 MB)


---
## 📋 Feature Engineering Summary

### IEEE-CIS

| Step | Action | Columns added / removed |
|------|--------|------------------------|
| Decode TransactionDT | Extract hour_of_day, day_of_week, day_of_month | +3 |
| Log-transform amount | log1p(TransactionAmt) | +1 |
| has_identity flag | Binary: matched in identity table | +1 |
| Missingness flags | Binary {col}_is_missing for V/id cols >30% NaN | +N |
| Drop >90% missing V-cols | Listed in cell 1.5 output | -M |
| Frequency encoding | card1_freq, addr1_freq, P_emaildomain_freq | +3 |
| Correlation pruning | Drop one of each pair with |r| > 0.95 | -K |
| Entity-graph features | card1_addr1_count, card1_addr1_email_count, amt_mean, amt_std | +4 |

**Final feature count:** See cell 1.10 output.

### PaySim

| Feature | Formula |
|---------|----------|
| balance_delta_orig | oldbalanceOrg - newbalanceOrig - amount |
| is_transfer | (type == 'TRANSFER') |
| is_cashout | (type == 'CASH_OUT') |
| orig_balance_drained | (newbalanceOrig == 0) |
| dest_balance_zero | (newbalanceDest == 0) |
| amount_to_balance_ratio | amount / (oldbalanceOrg + 1) |
| step_sin / step_cos | sin/cos(step × 2π / 24) |

**Final feature count:** 11 raw + 8 engineered = 19 columns  
(isFraud and isFlaggedFraud not included as modelling features)

---

### Decisions that deviate from or clarify the spec

1. **Entity-graph time window:**  
   The spec says *"count of other transactions sharing this card1+addr1 combo in the past N days."*  
   **Decision:** Global group counts (all training transactions with the same key) are used  
   instead of a time-windowed rolling count. Rationale: (a) the spec calls these features a  
   *proxy* for graph signal, (b) per-row time-window aggregation on 590K rows is significantly  
   more complex and slow, and (c) a real graph/GNN layer is explicitly scoped to a later phase.  
   Global counts still capture entity-level frequency signal without lookahead into val/test.

2. **No spec deviations on prohibited items:**  
   - No PCA or dimensionality reduction ✅  
   - Datasets kept separate ✅  
   - No device/IP columns in entity features ✅  

3. **FrequencyEncoder stores freq_maps_ as plain Python dicts** so they can be  
   `json.dump()`-serialised for the FastAPI scoring service without additional dependencies.

4. **Leakage prevention strategy:**  
   - Split happens *before* any fitting step.  
   - FrequencyEncoder, missingness rate computation, drop lists, and correlation matrix  
     are all computed on the training split only and applied via transform to val/test.  
   - Entity-graph aggregations use `train_df=ieee_train` to prevent val/test counts  
     leaking into the lookup table.


## ✅ Processed File Inventory

In [23]:
print(f'Files in {PROCESSED_DIR}:')
for f in sorted(os.listdir(PROCESSED_DIR)):
    full = os.path.join(PROCESSED_DIR, f)
    size_mb = os.path.getsize(full) / 1024**2
    print(f'  {f:<35s}  {size_mb:.1f} MB')


Files in c:\Users\DHANRAJ\fraudlens\data\processed:
  ieee_test.parquet                    14.8 MB
  ieee_train.parquet                   63.9 MB
  ieee_val.parquet                     14.0 MB
  paysim_test.parquet                  52.0 MB
  paysim_train.parquet                 241.8 MB
  paysim_val.parquet                   52.0 MB
